# Elenchus — serving the Socratic tutor from Colab

Runs the team's fine-tuned model on Colab's free GPU and puts a public address in
front of it, so anyone's VS Code extension can reach it with nothing installed.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

Then run both cells. The second prints an address and keeps running, holding the
session open. Paste that address into `endpoint.txt` on GitHub and the whole team
is connected.

Colab disconnects after a while and the address changes each restart. That is
what `endpoint.txt` is for: edit one line, everyone follows within five minutes.

In [ ]:
#@title 1. Set up and check (about two minutes)
import subprocess, sys, os, pathlib

def run(*args, **kw):
    return subprocess.run([sys.executable, "-m", "pip"] + list(args),
                          capture_output=True, text=True, **kw)

# The model, the server and the web page all live in the public HF Space repo.
# The Space itself has no compute quota, but it works well as somewhere to keep
# the files.
REPO = "https://huggingface.co/spaces/Alsalay/elenchus_poc"
if not pathlib.Path("/content/elenchus").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/elenchus"], check=True)
os.chdir("/content/elenchus")

# Colab ships torchao 0.10, and peft refuses to load its LoRA dispatcher when it
# finds a version below 0.16. We do not use torchao (it is for quantisation), and
# peft skips that code path entirely when it is absent, so remove it.
print("removing the incompatible torchao...")
run("uninstall", "-y", "-q", "torchao")
print("installing peft...")
r = run("install", "-q", "peft")
if r.returncode != 0:
    print(r.stdout[-2000:], r.stderr[-2000:])

# Fail here, with a readable message, rather than inside the server subprocess.
import importlib
for mod in ("torch", "transformers", "peft"):
    importlib.import_module(mod)
from peft import PeftConfig
cfg = PeftConfig.from_pretrained("out/adapter")

import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print()
print("GPU        :", gpu or "NONE — set Runtime to T4, or it will be slow")
print("base model :", cfg.base_model_name_or_path)
print("adapter    : loaded OK")
print("\nReady. Run the next cell.")

In [ ]:
#@title 2. Start the tutor and get its address
import subprocess, threading, queue, re, time, sys, urllib.request

PORT = 8008

def serve():
    server = subprocess.Popen(
        [sys.executable, "serve.py", "--host", "127.0.0.1", "--port", str(PORT)],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )

    print("loading the model (downloads ~1 GB the first time)...")
    for _ in range(120):
        if server.poll() is not None:
            print("\nThe server stopped before it was ready. Its output:\n")
            print(server.stdout.read())
            return
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{PORT}/v1/models", timeout=2).read()
            break
        except Exception:
            time.sleep(3)
    else:
        print("The server did not come up in time.")
        server.terminate()
        return
    print("model ready\n")

    # cloudflared prints its address once, into its own output, so it has to be
    # read as it goes rather than waited for.
    tunnel = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    found = queue.Queue()

    def watch():
        for line in tunnel.stdout:
            m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
            if m:
                found.put(m.group(0))

    threading.Thread(target=watch, daemon=True).start()
    try:
        url = found.get(timeout=90)
    except queue.Empty:
        print("cloudflared did not report an address.")
        tunnel.terminate(); server.terminate()
        return

    print("=" * 68)
    print("  THE TUTOR IS LIVE AT")
    print()
    print("     " + url + "/v1")
    print()
    print("  Put that line into endpoint.txt on GitHub:")
    print("  https://github.com/H7Feez/elenchus_poc/edit/main/endpoint.txt")
    print()
    print("  Everyone's extension picks it up within five minutes.")
    print("  Open " + url + " in a browser to try it without VS Code.")
    print("=" * 68)
    print("\nLeave this cell running. Replies appear below as people use it.\n")

    try:
        for line in server.stdout:
            print(line, end="")
    except KeyboardInterrupt:
        print("\nstopping")
    finally:
        tunnel.terminate()
        server.terminate()

# cloudflared runs here on Google's machines, so restrictions on tunnelling from
# your own connection do not apply.
import pathlib
if not pathlib.Path("/usr/local/bin/cloudflared").exists():
    print("fetching cloudflared...")
    subprocess.run(["wget", "-q", "-O", "/usr/local/bin/cloudflared",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
    subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"], check=True)

serve()